# Data Cleaning
Loads the datasets and cleans/makes them useable before saving them to the processed folder.

In [ ]:
import glob, os, json, ijson, sys, csv
sys.path.append("..")  # must come before utils import

import pandas as pd
from collections import defaultdict

from utils.cleaning import drop_irrelevant_columns, extract_tweet_text


### Cresci-15

In [53]:
RAW        = "../data/raw/cresci-2015"
OUT_USERS  = "../data/processed/cresci_2015/users_cresci_2015.csv"
OUT_TWEETS = "../data/processed/cresci_2015/tweets_cresci_2015.csv"

SUBSETS = {
    "TFP": "human",
    "TWT": "human",
    "FSF": "bot",
    "INT": "bot",
    "E13": "bot",
}

print("Paths set. Subsets to load:", list(SUBSETS.keys()))

Paths set. Subsets to load: ['TFP', 'TWT', 'FSF', 'INT', 'E13']


In [54]:
# get the number of users and load them. Find basic data shape

user_frames = []

for subset, label in SUBSETS.items():
    path = f"{RAW}/{subset}.csv/users.csv"
    df = pd.read_csv(path)
    df["label"]  = label
    df["subset"] = subset
    user_frames.append(df)
    print(f"  {subset}: {len(df)} users ({label})")

users = pd.concat(user_frames, ignore_index=True)
print(f"\nTotal users loaded: {len(users)}")
print(f"Label distribution:\n{users['label'].value_counts().to_string()}")

  TFP: 469 users (human)
  TWT: 845 users (human)
  FSF: 1169 users (bot)
  INT: 1337 users (bot)
  E13: 1481 users (bot)

Total users loaded: 5301
Label distribution:
label
bot      3987
human    1314


In [55]:
# Keep raw columns needed for feature engineering later
KEEP = [
    "id", "screen_name", "label", "subset",
    "followers_count", "friends_count", "statuses_count",
    "favourites_count", "listed_count", "created_at",
    "verified", "description", "location", "default_profile_image",
]

users = drop_irrelevant_columns(users, KEEP)
users = users.rename(columns={"id": "user_id"})

Kept 14 columns, dropped 22
Dropped: ['name', 'url', 'lang', 'time_zone', 'default_profile', 'geo_enabled', 'profile_image_url', 'profile_banner_url', 'profile_use_background_image', 'profile_background_image_url_https', 'profile_text_color', 'profile_image_url_https', 'profile_sidebar_border_color', 'profile_background_tile', 'profile_sidebar_fill_color', 'profile_background_image_url', 'profile_background_color', 'profile_link_color', 'utc_offset', 'protected', 'updated', 'dataset']
Remaining nulls:
verified                 5301
description              1216
location                 1212
default_profile_image    5276


In [56]:
# Load raw tweet text for all subsets
tweet_frames = []

for subset in SUBSETS:
    path = f"{RAW}/{subset}.csv/tweets.csv"
    df = pd.read_csv(path, low_memory=False, encoding="latin-1")
    tweet_frames.append(df)
    print(f"  {subset}: {len(df)} tweets")

raw_tweets = pd.concat(tweet_frames, ignore_index=True)
print(f"\nTotal tweets loaded: {len(raw_tweets):,}")

tweets = extract_tweet_text(raw_tweets, users["user_id"])

  TFP: 563693 tweets
  TWT: 114192 tweets
  FSF: 22910 tweets
  INT: 58925 tweets
  E13: 2068037 tweets

Total tweets loaded: 2,827,757
Tweets from labeled users: 2,827,757 / 2,827,757 total


In [ ]:
# Save the files
os.makedirs(os.path.dirname(OUT_USERS), exist_ok=True)
users["source"] = "cresci_2015"
users.to_csv(OUT_USERS, index=False)
tweets.to_csv(OUT_TWEETS, index=False)

print(f"Users  → {OUT_USERS}  {users.shape}")
print(f"Tweets → {OUT_TWEETS}  {tweets.shape}")
print(f"User columns:  {list(users.columns)}")
print(f"Tweet columns: {list(tweets.columns)}")
print(f"Label distribution: {users["label"].value_counts().to_string()}")


### Cresci-17

In [58]:
RAW_17        = "../data/raw/cresci-2017"
OUT_USERS_17  = "../data/processed/cresci_2017/users_cresci_2017.csv"
OUT_TWEETS_17 = "../data/processed/cresci_2017/tweets_cresci_2017.csv"

SUBSETS_17 = {
    "genuine_accounts.csv":       "human",
    "fake_followers.csv":         "bot",
    "social_spambots_1.csv":      "bot",
    "social_spambots_2.csv":      "bot",
    "social_spambots_3.csv":      "bot",
    "traditional_spambots_1.csv": "bot",
    "traditional_spambots_2.csv": "bot",
    "traditional_spambots_3.csv": "bot",
    "traditional_spambots_4.csv": "bot",
}

NO_TWEETS_17 = {"traditional_spambots_2.csv", "traditional_spambots_3.csv", "traditional_spambots_4.csv"}

print("Paths set. Subsets to load:", list(SUBSETS_17.keys()))

Paths set. Subsets to load: ['genuine_accounts.csv', 'fake_followers.csv', 'social_spambots_1.csv', 'social_spambots_2.csv', 'social_spambots_3.csv', 'traditional_spambots_1.csv', 'traditional_spambots_2.csv', 'traditional_spambots_3.csv', 'traditional_spambots_4.csv']


In [59]:
# load the users and label
user_frames_17 = []

for subset, label in SUBSETS_17.items():
    path = f"{RAW_17}/{subset}/users.csv"
    df = pd.read_csv(path, encoding="latin-1")
    df["label"]  = label
    df["subset"] = subset
    user_frames_17.append(df)
    print(f"  {subset}: {len(df)} users ({label})")

users_17 = pd.concat(user_frames_17, ignore_index=True)
print(f"\nTotal users loaded: {len(users_17)}")
print(f"Label distribution:\n{users_17['label'].value_counts().to_string()}")

  genuine_accounts.csv: 3474 users (human)
  fake_followers.csv: 3351 users (bot)
  social_spambots_1.csv: 991 users (bot)
  social_spambots_2.csv: 3457 users (bot)
  social_spambots_3.csv: 464 users (bot)
  traditional_spambots_1.csv: 1000 users (bot)
  traditional_spambots_2.csv: 100 users (bot)
  traditional_spambots_3.csv: 403 users (bot)
  traditional_spambots_4.csv: 1128 users (bot)

Total users loaded: 14368
Label distribution:
label
bot      10894
human     3474


In [60]:
# drop columns
KEEP_17 = [
    "id", "screen_name", "label", "subset",
    "followers_count", "friends_count", "statuses_count",
    "favourites_count", "listed_count", "created_at",
    "verified", "description", "location", "default_profile_image",
]

users_17 = drop_irrelevant_columns(users_17, KEEP_17)
users_17 = users_17.rename(columns={"id": "user_id"})

Kept 14 columns, dropped 30
Dropped: ['name', 'url', 'lang', 'time_zone', 'default_profile', 'geo_enabled', 'profile_image_url', 'profile_banner_url', 'profile_use_background_image', 'profile_background_image_url_https', 'profile_text_color', 'profile_image_url_https', 'profile_sidebar_border_color', 'profile_background_tile', 'profile_sidebar_fill_color', 'profile_background_image_url', 'profile_background_color', 'profile_link_color', 'utc_offset', 'is_translator', 'follow_request_sent', 'protected', 'notifications', 'contributors_enabled', 'following', 'timestamp', 'crawled_at', 'updated', 'test_set_1', 'test_set_2']
Remaining nulls:
verified                 14357
description               5744
location                  6684
default_profile_image    14290


In [61]:
# load the tweet text
tweet_frames_17 = []

for subset in SUBSETS_17:
    if subset in NO_TWEETS_17:
        print(f"  {subset}: no tweets.csv — skipping")
        continue
    path = f"{RAW_17}/{subset}/tweets.csv"
    df = pd.read_csv(path, low_memory=False, encoding="latin-1")
    tweet_frames_17.append(df)
    print(f"  {subset}: {len(df)} tweets")

raw_tweets_17 = pd.concat(tweet_frames_17, ignore_index=True)
print(f"\nTotal tweets loaded: {len(raw_tweets_17):,}")

tweets_17 = extract_tweet_text(raw_tweets_17, users_17["user_id"])

In [ ]:
# save
os.makedirs(os.path.dirname(OUT_USERS_17), exist_ok=True)
users_17["source"] = "cresci_2017"
users_17.to_csv(OUT_USERS_17, index=False)
tweets_17.to_csv(OUT_TWEETS_17, index=False)

print(f"Users  → {OUT_USERS_17}  {users_17.shape}")
print(f"Tweets → {OUT_TWEETS_17}  {tweets_17.shape}")
print(f"User columns:  {list(users_17.columns)}")
print(f"Tweet columns: {list(tweets_17.columns)}")
print(f"Label distribution: {users_17["label"].value_counts().to_string()}")


### Twibot-20

In [81]:
RAW_20        = "../data/raw/Twibot20"
OUT_USERS_20  = "../data/processed/twibot_2020/users_twibot_20.csv"
OUT_TWEETS_20 = "../data/processed/twibot_2020/tweets_twibot_20.csv"

SPLITS_20 = ["train", "test", "dev"]
LABEL_MAP = {"0": "human", "1": "bot"}

print("Paths set. Splits to load:", SPLITS_20)

Paths set. Splits to load: ['train', 'test', 'dev']


In [73]:
# loading records
user_records  = []
tweet_records = []

for split in SPLITS_20:
    with open(f"{RAW_20}/{split}.json") as f:
        data = json.load(f)

    for rec in data:
        uid   = str(rec["ID"])
        label = LABEL_MAP[str(rec["label"])]

        user_records.append({**rec["profile"], "user_id": uid, "label": label, "subset": split})

        if rec["tweet"] is not None:
            for text in rec["tweet"]:
                tweet_records.append({"user_id": uid, "text": text})

    print(f"  {split}: {len(data)} users")

print(f"\nTotal users:  {len(user_records):,}")
print(f"Total tweets: {len(tweet_records):,}")

  train: 8278 users
  test: 1183 users
  dev: 2365 users

Total users:  11,826
Total tweets: 1,999,788


In [76]:
# drop columns
users_20 = pd.DataFrame(user_records)

# profile already has v1 flat fields — rename id to user_id if it exists as a separate column
if "id" in users_20.columns:
    users_20 = users_20.drop(columns=["id"])

print(f"Available profile columns: {list(users_20.columns)}")
print(f"\nLabel distribution:\n{users_20['label'].value_counts().to_string()}")
print(f"Split distribution:\n{users_20['subset'].value_counts().to_string()}")

KEEP_20 = [
    "user_id", "screen_name", "label", "subset",
    "followers_count", "friends_count", "statuses_count",
    "favourites_count", "listed_count", "created_at",
    "verified", "description", "location", "default_profile_image",
]

users_20 = drop_irrelevant_columns(users_20, KEEP_20)

Available profile columns: ['id_str', 'name', 'screen_name', 'location', 'profile_location', 'description', 'url', 'entities', 'protected', 'followers_count', 'friends_count', 'listed_count', 'created_at', 'favourites_count', 'utc_offset', 'time_zone', 'geo_enabled', 'verified', 'statuses_count', 'lang', 'contributors_enabled', 'is_translator', 'is_translation_enabled', 'profile_background_color', 'profile_background_image_url', 'profile_background_image_url_https', 'profile_background_tile', 'profile_image_url', 'profile_image_url_https', 'profile_link_color', 'profile_sidebar_border_color', 'profile_sidebar_fill_color', 'profile_text_color', 'profile_use_background_image', 'has_extended_profile', 'default_profile', 'default_profile_image', 'user_id', 'label', 'subset']

Label distribution:
label
bot      6589
human    5237
Split distribution:
subset
train    8278
dev      2365
test     1183
Kept 14 columns, dropped 26
Dropped: ['id_str', 'name', 'profile_location', 'url', 'entities',

In [77]:
# tweets
tweets_20 = pd.DataFrame(tweet_records)
tweets_20["user_id"] = tweets_20["user_id"].astype(str)

# Filter to only labeled users (exclude any neighbor bleed-through)
tweets_20 = tweets_20[tweets_20["user_id"].isin(users_20["user_id"].astype(str))]

print(f"Tweet rows: {len(tweets_20):,}")
print(f"Unique users with tweets: {tweets_20['user_id'].nunique():,}")
print(f"Columns: {list(tweets_20.columns)}")
print("Note: no tweet_id or created_at — original format stores tweets as plain strings only")

Tweet rows: 1,999,788
Unique users with tweets: 11,746
Columns: ['user_id', 'text']
Note: no tweet_id or created_at — original format stores tweets as plain strings only


In [ ]:
# save
os.makedirs(os.path.dirname(OUT_USERS_20), exist_ok=True)
users_20["source"] = "twibot_20"
users_20.to_csv(OUT_USERS_20, index=False)
tweets_20.to_csv(OUT_TWEETS_20, index=False)

print(f"Users  → {OUT_USERS_20}  {users_20.shape}")
print(f"Tweets → {OUT_TWEETS_20}  {tweets_20.shape}")
print(f"User columns:  {list(users_20.columns)}")
print(f"Tweet columns: {list(tweets_20.columns)}")
print(f"Label distribution:{users_20["label"].value_counts().to_string()}")


### Twibot-22

In [20]:
RAW_22        = "../data/raw/Twibot22"
OUT_USERS_22  = "../data/processed/twibot_2022/users_twibot_22.csv"
OUT_TWEETS_22 = "../data/processed/twibot_2022/tweets_twibot_22.csv"

print("Paths set.")

Paths set.


In [ ]:
# Load labels, splits, and users
labels_22 = pd.read_csv(f"{RAW_22}/label.csv")
labels_22.columns = ["user_id", "label"]
labels_22["user_id"] = labels_22["user_id"].astype(str)

splits_22 = pd.read_csv(f"{RAW_22}/split.csv")
splits_22.columns = ["user_id", "subset"]
splits_22["user_id"] = splits_22["user_id"].astype(str)

labeled_ids = set(labels_22["user_id"])
print(f"Labeled users: {len(labeled_ids):,}")
print(f"Label distribution:\n{labels_22['label'].value_counts().to_string()}")
print(f"\nSplit distribution:\n{splits_22['subset'].value_counts().to_string()}")

print("\nLoading user.json...")
with open(f"{RAW_22}/user.json") as f:
    raw_users = json.load(f)
print(f"Total users in user.json: {len(raw_users):,}")

Labeled users: 1,000,000
Label distribution:
label
human    860057
bot      139943

Split distribution:
subset
train    700000
val      200000
test     100000

Loading user.json...
Total users in user.json: 1,000,000


In [ ]:
# build dataframe
user_records = []

for u in raw_users:
    uid = str(u["id"])
    if uid not in labeled_ids:
        continue

    pm = u.get("public_metrics") or {}
    user_records.append({
        "user_id":               uid,
        "screen_name":           u.get("username"),
        "followers_count":       pm.get("followers_count"),
        "friends_count":         pm.get("following_count"),   # normalize name
        "statuses_count":        pm.get("tweet_count"),       # normalize name
        "listed_count":          pm.get("listed_count"),
        "favourites_count":      None,                        # not in API v2
        "created_at":            u.get("created_at"),
        "verified":              u.get("verified"),
        "description":           u.get("description"),
        "location":              u.get("location"),
        "default_profile_image": int("default_profile" in str(u.get("profile_image_url", ""))),
    })

users_22 = pd.DataFrame(user_records)
users_22 = users_22.merge(labels_22, on="user_id", how="left")
users_22 = users_22.merge(splits_22, on="user_id", how="left")

print(f"Labeled users extracted: {len(users_22):,}")
print(f"Remaining nulls:\n{users_22.isnull().sum()[users_22.isnull().sum() > 0].to_string()}")

Labeled users extracted: 1,000,000
Remaining nulls:
favourites_count    1000000
location             291542


In [ ]:
# Stream tweets directly to CSV — never accumulates in RAM
tweet_shards = sorted(glob.glob(f"{RAW_22}/tweet_*.json"))
print(f"Tweet shards found: {len(tweet_shards)}")

os.makedirs(os.path.dirname(OUT_TWEETS_22), exist_ok=True)

total_collected = 0

with open(OUT_TWEETS_22, "w", encoding="utf-8", newline="") as csv_out:
    writer = csv.DictWriter(csv_out,
                            fieldnames=["tweet_id", "user_id", "text", "created_at"])
    writer.writeheader()

    for shard_path in tweet_shards:
        shard_name      = os.path.basename(shard_path)
        shard_collected = 0

        with open(shard_path, "rb") as f:
            for rec in ijson.items(f, "item"):
                uid = "u" + str(rec["author_id"])
                if uid not in labeled_ids:
                    continue
                writer.writerow({
                    "tweet_id":   str(rec["id"]),
                    "user_id":    uid,
                    "text":       rec.get("text", ""),
                    "created_at": rec.get("created_at", ""),
                })
                shard_collected += 1
                total_collected += 1

        print(f"  {shard_name}: {shard_collected:,} tweets  "
              f"(running total: {total_collected:,})")

print(f"\nTotal tweets written: {total_collected:,}")

In [ ]:
# tweets already written to disk in previous cell
os.makedirs(os.path.dirname(OUT_USERS_22), exist_ok=True)
users_22["source"] = "twibot_22"
users_22.to_csv(OUT_USERS_22, index=False)

tweet_row_count = sum(1 for _ in open(OUT_TWEETS_22, encoding="utf-8")) - 1

print(f"Users  → {OUT_USERS_22}  {users_22.shape}")
print(f"Tweets → {OUT_TWEETS_22}  ({tweet_row_count:,} rows)")
print(f"\nUser columns:  {list(users_22.columns)}")
print(f"\nLabel distribution:\n{users_22['label'].value_counts().to_string()}")